<a href="https://colab.research.google.com/github/harshitasdev8/Myeloid-Project-Oda-Lab/blob/PDAC_MultiGenes/PDAC_Working.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
"""
PDAC Myeloid Score — Survival Analysis Pipeline (v2: composite scores)
========================================================================
Data source: UCSC Xena, TCGA Pancreatic Cancer (PAAD)

INSTRUCTIONS FOR YOU:
1. Change RAW_PATH below to wherever your new TSV file is saved on your computer.
2. Make sure you have 'lifelines' and 'pandas' installed. If not, run this first
   in a notebook cell:  !pip install lifelines pandas
3. Run all cells / run the whole script top to bottom.
4. All results print to the screen AND get saved as CSV files you can open in
   a spreadsheet (see the bottom of the script for exact filenames).

GENES USED, GROUPED INTO AXES:
- Macrophage abundance : CD68, CD14
- M1 (pro-inflammatory) : CD80, CD86, HLA-DRA, NOS2
- M2 (suppressive)      : CD163, MRC1, MSR1
- CD8 T cells           : CD8A, CD8B
- CD40 (gate)           : CD40

If you weren't able to pull one of these genes in Xena (e.g. HLA-DRA didn't
match), just delete it from the relevant list below — the script will still
run fine with fewer genes in that axis.
"""

"\nPDAC Myeloid Score — Survival Analysis Pipeline (v2: composite scores)\n========================================================================\nData source: UCSC Xena, TCGA Pancreatic Cancer (PAAD)\n \nINSTRUCTIONS FOR YOU:\n1. Change RAW_PATH below to wherever your new TSV file is saved on your computer.\n2. Make sure you have 'lifelines' and 'pandas' installed. If not, run this first\n   in a notebook cell:  !pip install lifelines pandas\n3. Run all cells / run the whole script top to bottom.\n4. All results print to the screen AND get saved as CSV files you can open in\n   a spreadsheet (see the bottom of the script for exact filenames).\n \nGENES USED, GROUPED INTO AXES:\n- Macrophage abundance : CD68, CD14\n- M1 (pro-inflammatory) : CD80, CD86, HLA-DRA, NOS2\n- M2 (suppressive)      : CD163, MRC1, MSR1\n- CD8 T cells           : CD8A, CD8B\n- CD40 (gate)           : CD40\n \nIf you weren't able to pull one of these genes in Xena (e.g. HLA-DRA didn't\nmatch), just delete it fr

In [14]:
pip install lifelines

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import pandas as pd
from lifelines import CoxPHFitter
from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------------------------
# STEP 0: EDIT THIS PATH to point to your new TSV file
# ---------------------------------------------------------------------------
RAW_PATH = "/content/denseDataOnlyDownload_PDAC_noNOS2_noCD80.tsv"   # <-- change if needed

# ---------------------------------------------------------------------------
# STEP 1: Define which genes belong to which axis
#          Edit these lists if any gene didn't pull through in Xena.
# ---------------------------------------------------------------------------
ABUNDANCE_GENES = ["CD68", "CD14"]
M1_GENES        = ["CD86", "HLA-DRA"]
M2_GENES        = ["CD163", "MRC1", "MSR1"]
CD8_GENES       = ["CD8A", "CD8B"]
CD40_GENE       = ["CD40"]

ALL_GENES = ABUNDANCE_GENES + M1_GENES + M2_GENES + CD8_GENES + CD40_GENE


In [17]:


import os

# Define the Google Drive folder path
drive_output_dir = "/content/drive/MyDrive/Colab Notebooks/PDAC_MultiGenes_Data"

# Create the directory if it doesn't exist
os.makedirs(drive_output_dir, exist_ok=True)

In [18]:
# ---------------------------------------------------------------------------
# STEP 2: Load raw data
# ---------------------------------------------------------------------------
df = pd.read_csv(RAW_PATH, sep="\t")

# Keep only genes that actually exist in your file (in case you dropped one)
ALL_GENES = [g for g in ALL_GENES if g in df.columns]
ABUNDANCE_GENES = [g for g in ABUNDANCE_GENES if g in df.columns]
M1_GENES = [g for g in M1_GENES if g in df.columns]
M2_GENES = [g for g in M2_GENES if g in df.columns]
CD8_GENES = [g for g in CD8_GENES if g in df.columns]
CD40_GENE = [g for g in CD40_GENE if g in df.columns]

print("Genes actually found in your file:", ALL_GENES)


Genes actually found in your file: ['CD68', 'CD14', 'CD86', 'HLA-DRA', 'CD163', 'MRC1', 'MSR1', 'CD8A', 'CD8B', 'CD40']


In [19]:
# ---------------------------------------------------------------------------
# STEP 3: Clean — keep only primary tumor samples (TCGA code "01")
# ---------------------------------------------------------------------------
df["sample_type_code"] = df["sample"].str.split("-").str[-1]
df_tumor = df[df["sample_type_code"] == "01"].copy()
print(f"Samples after filtering to primary tumor only: {df_tumor.shape[0]}")

df_tumor = df_tumor.dropna(subset=ALL_GENES)
print(f"Samples after dropping missing gene expression: {df_tumor.shape[0]}")


Samples after filtering to primary tumor only: 185
Samples after dropping missing gene expression: 178


In [20]:
# ---------------------------------------------------------------------------
# STEP 4: Build survival time + event indicator
# ---------------------------------------------------------------------------
df_tumor["event"] = (df_tumor["vital_status"] == "DECEASED").astype(int)
df_tumor["time"] = df_tumor["days_to_death"].fillna(df_tumor["days_to_last_followup"])
assert df_tumor["time"].isna().sum() == 0, "Some patients have no survival time — check raw data"

print(f"\nFinal analysis cohort: {df_tumor.shape[0]} patients "
      f"({df_tumor['event'].sum()} deaths, {(df_tumor['event']==0).sum()} censored)")


Final analysis cohort: 178 patients (93 deaths, 85 censored)


In [21]:
 #---------------------------------------------------------------------------
# STEP 5: Build composite scores
#          Each composite = simple average of its genes' expression values.
#          (Gene expression here is already log2-scale, so averaging is standard.)
# ---------------------------------------------------------------------------
if ABUNDANCE_GENES:
    df_tumor["abundance_score"] = df_tumor[ABUNDANCE_GENES].mean(axis=1)
if M1_GENES:
    df_tumor["M1_score"] = df_tumor[M1_GENES].mean(axis=1)
if M2_GENES:
    df_tumor["M2_score"] = df_tumor[M2_GENES].mean(axis=1)
if CD8_GENES:
    df_tumor["CD8_score"] = df_tumor[CD8_GENES].mean(axis=1)
if CD40_GENE:
    df_tumor["CD40_score"] = df_tumor[CD40_GENE].mean(axis=1)  # just CD40 itself

# M1:M2 ratio (log-scale subtraction = ratio)
if M1_GENES and M2_GENES:
    df_tumor["M1_M2_ratio"] = df_tumor["M1_score"] - df_tumor["M2_score"]

# M2:CD8 "suppression wall" ratio
if M2_GENES and CD8_GENES:
    df_tumor["M2_CD8_ratio"] = df_tumor["M2_score"] - df_tumor["CD8_score"]

COMPOSITE_VARS = [c for c in
                   ["abundance_score", "M1_score", "M2_score", "CD8_score",
                    "CD40_score", "M1_M2_ratio", "M2_CD8_ratio"]
                   if c in df_tumor.columns]


In [22]:

# ---------------------------------------------------------------------------
# STEP 6a: Univariate Cox models — each composite score tested alone
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("UNIVARIATE COX MODELS (each composite score tested alone)")
print("=" * 70)

univariate_results = []
for var in COMPOSITE_VARS:
    cph = CoxPHFitter()
    data = df_tumor[[var, "time", "event"]].dropna()
    cph.fit(data, duration_col="time", event_col="event")
    row = cph.summary.loc[var]
    univariate_results.append({
        "variable": var,
        "HR": row["exp(coef)"],
        "p_raw": row["p"],
        "C-index": cph.concordance_index_,
        "n_patients": data.shape[0]
    })

univ_df = pd.DataFrame(univariate_results)

# Multiple-testing correction (Benjamini-Hochberg / FDR) since we're testing
# several variables at once — this controls for finding "significant" results
# by chance alone.
univ_df["p_adj_BH"] = multipletests(univ_df["p_raw"], method="fdr_bh")[1]
print(univ_df.to_string(index=False))

# ---------------------------------------------------------------------------
# STEP 6b: Multivariate Cox model — abundance, M1, M2, CD8, CD40 together
#          This tests whether each score predicts survival INDEPENDENTLY
#          of the others.
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("MULTIVARIATE COX MODEL (abundance + M1 + M2 + CD8 + CD40 together)")
print("=" * 70)

core_vars = [v for v in ["abundance_score", "M1_score", "M2_score", "CD8_score", "CD40_score"]
             if v in df_tumor.columns]
cph_multi = CoxPHFitter()
cph_multi.fit(df_tumor[core_vars + ["time", "event"]], duration_col="time", event_col="event")
print(cph_multi.summary[["coef", "exp(coef)", "p"]].to_string())
print(f"\nConcordance index: {cph_multi.concordance_index_:.3f}")

# ---------------------------------------------------------------------------
# STEP 6c: Multivariate model using ratios instead of raw scores
#          (M1:M2 ratio + M2:CD8 ratio + CD40, controlling for abundance)
# ---------------------------------------------------------------------------
if "M1_M2_ratio" in df_tumor.columns and "M2_CD8_ratio" in df_tumor.columns:
    print("\n" + "=" * 70)
    print("MULTIVARIATE COX MODEL (M1:M2 ratio + M2:CD8 ratio + CD40 + abundance)")
    print("=" * 70)
    ratio_vars = [v for v in ["abundance_score", "M1_M2_ratio", "M2_CD8_ratio", "CD40_score"]
                  if v in df_tumor.columns]
    cph_ratio = CoxPHFitter()
    cph_ratio.fit(df_tumor[ratio_vars + ["time", "event"]], duration_col="time", event_col="event")
    print(cph_ratio.summary[["coef", "exp(coef)", "p"]].to_string())
    print(f"\nConcordance index: {cph_ratio.concordance_index_:.3f}")

# ---------------------------------------------------------------------------
# STEP 7: Save everything as CSV files you can open in a spreadsheet
# ---------------------------------------------------------------------------
df_tumor.to_csv("pdac_clean_with_composite_scores.csv", index=False)
univ_df.to_csv("pdac_univariate_composite_results.csv", index=False)
cph_multi.summary.to_csv("pdac_multivariate_core_results.csv")
if "M1_M2_ratio" in df_tumor.columns and "M2_CD8_ratio" in df_tumor.columns:
    cph_ratio.summary.to_csv("pdac_multivariate_ratio_results.csv")

print("\nSaved output files:")
print("  - pdac_clean_with_composite_scores.csv  (full cleaned data + all scores)")
print("  - pdac_univariate_composite_results.csv (each score tested alone, with FDR-corrected p-values)")
print("  - pdac_multivariate_core_results.csv    (all scores tested together)")
print("  - pdac_multivariate_ratio_results.csv   (ratio-based model)")


UNIVARIATE COX MODELS (each composite score tested alone)
       variable       HR    p_raw  C-index  n_patients  p_adj_BH
abundance_score 1.193449 0.149943 0.495030         178  0.349867
       M1_score 1.131280 0.146941 0.499145         178  0.349867
       M2_score 1.066406 0.371599 0.503260         178  0.590213
      CD8_score 1.021747 0.759008 0.476112         178  0.759008
     CD40_score 1.290705 0.018678 0.535859         178  0.130748
    M1_M2_ratio 1.117267 0.425759 0.521911         178  0.590213
   M2_CD8_ratio 1.054087 0.505897 0.524156         178  0.590213

MULTIVARIATE COX MODEL (abundance + M1 + M2 + CD8 + CD40 together)
                     coef  exp(coef)         p
covariate                                     
abundance_score -0.054805   0.946670  0.834477
M1_score         0.091050   1.095324  0.720943
M2_score         0.005573   1.005589  0.971897
CD8_score       -0.140640   0.868802  0.176522
CD40_score       0.317575   1.373792  0.040721

Concordance index: 0.56

### Consolidating results into an Excel file

In [23]:
output_excel_path = os.path.join(drive_output_dir, "pdac_all_results.xlsx")

with pd.ExcelWriter(output_excel_path) as writer:
    # Load and write each CSV to a different sheet
    df_tumor_clean = pd.read_csv("pdac_clean_with_composite_scores.csv")
    df_tumor_clean.to_excel(writer, sheet_name='Cleaned Data', index=False)

    df_univariate = pd.read_csv("pdac_univariate_composite_results.csv")
    df_univariate.to_excel(writer, sheet_name='Univariate Results', index=False)

    df_multivariate_core = pd.read_csv("pdac_multivariate_core_results.csv", index_col=0)
    df_multivariate_core.to_excel(writer, sheet_name='Multivariate Core Results')

    if os.path.exists("pdac_multivariate_ratio_results.csv"):
        df_multivariate_ratio = pd.read_csv("pdac_multivariate_ratio_results.csv", index_col=0)
        df_multivariate_ratio.to_excel(writer, sheet_name='Multivariate Ratio Results')

print(f"All results saved to Excel file: {output_excel_path}")

All results saved to Excel file: /content/drive/MyDrive/Colab Notebooks/PDAC_MultiGenes_Data/pdac_all_results.xlsx
